# 1D J1J2 Inference: LorentzGRU (seed 111-555) - part 1

This is part of the work arxiv:2604.24337 [quant-ph] by HL Dao. For the purpose of reproducing the results, the paths to the trained weights have to be adjusted to match the repo structure.

In this notebook, the inferences were done for J2=0.0, 0.2, 0.5. In another notebook, the J2=0.8 case inference is done. 

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import time
sys.path.append('../../1_hypnqs_torch_lorentz/utility_lorentz')
from j1j2_train_loop_lorentz import *

Hypercore Lorentzian module loaded successfully with Geoopt wrappers.
GPU Available:  False


In [2]:
def set_cpu_deterministic(seed):
    # 1. Python & Numpy
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # 2. PyTorch CPU
    torch.manual_seed(seed)
    
    # 3. Force Deterministic Algorithms
    # This prevents non-deterministic CPU operations (like some views/reductions)
    torch.use_deterministic_algorithms(True)
    
    # 4. Limit CPU Threads
    # Setting this to 1 ensures operations are done in a fixed order.
    torch.set_num_threads(1)

In [3]:
def clip_local_energies(eloc, threshold=5.0):
    # Convert to numpy if it's a torch tensor, or vice versa
    eloc_real = np.real(eloc)
    median = np.median(eloc_real)
    mad = np.median(np.abs(eloc_real - median))
    
    # Standard safety check to avoid division by zero if MAD is 0
    if mad == 0:
        return eloc
        
    lower_bound = median - threshold * mad
    upper_bound = median + threshold * mad
    
    # Clip the values (keeping the imaginary part if it exists)
    # We create a copy to avoid modifying the original array in place
    clipped = np.clip(eloc_real, lower_bound, upper_bound)
    
    # If the original was complex, restore the imaginary part
    if np.iscomplexobj(eloc):
        return clipped + 1j * np.imag(eloc)
    return clipped  
    
def define_load_test_no_tau(wf,  numsamples,path_to_weights, Ee, clipped_e = False):
    state_dict = torch.load(path_to_weights, map_location=torch.device('cpu'))   
    new_state_dict = {}
    for key, value in state_dict.items():
        # Strip prefixes and rename keys to match current architecture
        new_key = key.replace('model.', '').replace('cell.', 'rnn.')
        new_state_dict[new_key] = value
    #  Load the RE-MAPPED weights
    wf.model.load_state_dict(new_state_dict, strict=False)
    print("Successfully remapped and loaded weights.")
    
    with torch.no_grad():
        test_samples_after = wf.sample_no_tau(numsamples)
        if clipped_e:
            # 1. Get raw energies
            raw_gs_after = J1J2_local_energies(wf, N, J1, J2, Bz, numsamples, test_samples_after, Marshall_sign=True)
    
            # 2. APPLY CLIPPING
            test_gs_after = clip_local_energies(raw_gs_after, threshold=5.0)
    
            # 3. Calculate statistics on cleaned data
            gs_mean_a = np.round(np.mean(test_gs_after), 4)
            gs_var_a = np.round(np.var(test_gs_after), 4)
    
            # Optional: Count how many were clipped to see if the model is unstable
            num_clipped = np.sum(np.real(raw_gs_after) != np.real(test_gs_after))
            print(f"Clipped {num_clipped} outlier samples out of {numsamples}")
        else:
            test_gs_after =  J1J2_local_energies(wf, N, J1, J2, Bz, numsamples, test_samples_after, Marshall_sign=True)
            gs_mean_a = np.round(np.mean(test_gs_after),4)
            gs_var_a = np.round(np.var(test_gs_after),4)
    
    #wf.model.summary()
    #print('====================================================================')
    print(f'After loading weights, the ground state energy mean and variance are:')
    print(f'Mean E = {gs_mean_a}, var E = {gs_var_a}')
    print(f'DMRG energy  is {np.round(Ee,4)}')

In [4]:
N=100
numsamples = 10000
units = 70
syssize = 100
J1_ = 1.0
J1 = +J1_ * np.ones(syssize)
Bz = +0.0 * np.ones(syssize)
fname=f'results_LorentzGRU'

## J2=0.0, spatial clamp = 4.0

In [9]:
J2_ = 0.0
J2 = +J2_ * np.ones(syssize)
Ee_00=-44.12773986967251
spatial_clamp=4.0

In [11]:
seed=111
set_cpu_deterministic(seed)
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzGRU', units=units,spatial_clamp=4.0, seed=seed)
h_wl = f'J2={J2_}_sc={spatial_clamp}/seed={seed}/N100_J1=1.0|J2={J2_}_100_LorentzGRU_70_ns=80_MsTrue_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf, numsamples, h_wl, Ee_00, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 15,615
Successfully remapped and loaded weights.
Clipped 273 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-43.81570053100586+0.00139999995008111j), var E = 0.24089999496936798
DMRG energy  is -44.1277
Time taken=0.76 hrs


In [8]:
seed=222
set_cpu_deterministic(seed)
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzGRU', units=units,spatial_clamp=4.0, seed=seed)
h_wl = f'{fname}/J2={J2_}_sc={spatial_clamp}/seed={seed}/N100_J1=1.0|J2={J2_}_100_LorentzGRU_70_ns=80_MsTrue_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf, numsamples, h_wl, Ee_00, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 15,615
Successfully remapped and loaded weights.
Clipped 112 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-43.21189880371094-0.008799999952316284j), var E = 0.4535999894142151
DMRG energy  is -44.1277
Time taken=1.057 hrs


In [10]:
seed=333
set_cpu_deterministic(seed)
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzGRU', units=units,spatial_clamp=4.0, seed=seed)
h_wl = f'{fname}/J2={J2_}_sc={spatial_clamp}/seed={seed}/N100_J1=1.0|J2={J2_}_100_LorentzGRU_70_ns=80_MsTrue_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf, numsamples, h_wl, Ee_00, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 15,615
Successfully remapped and loaded weights.
Clipped 185 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-42.744598388671875+0.004100000020116568j), var E = 0.6848000288009644
DMRG energy  is -44.1277
Time taken=1.097 hrs


In [6]:
seed=444
set_cpu_deterministic(seed)
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzGRU', units=units,spatial_clamp=4.0, seed=seed)
h_wl = f'J2={J2_}_sc={spatial_clamp}/seed={seed}/N100_J1=1.0|J2={J2_}_100_LorentzGRU_70_ns=80_MsTrue_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf, numsamples, h_wl, Ee_00, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 15,615
Successfully remapped and loaded weights.
Clipped 496 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-43.71730041503906+0.00419999985024333j), var E = 0.21809999644756317
DMRG energy  is -44.1277
Time taken=0.362 hrs


In [6]:
seed=555
set_cpu_deterministic(seed)
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzGRU', units=units,spatial_clamp=4.0, seed=seed)
h_wl = f'J2={J2_}_sc={spatial_clamp}/seed={seed}/N100_J1=1.0|J2={J2_}_100_LorentzGRU_70_ns=80_MsTrue_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf, numsamples, h_wl, Ee_00, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 15,615
Successfully remapped and loaded weights.
Clipped 412 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-43.98820114135742-0.002300000051036477j), var E = 0.14309999346733093
DMRG energy  is -44.1277
Time taken=0.919 hrs


## J2=0.2, spatial clamp = 6.0

In [12]:
J2_ = 0.2
J2 = +J2_ * np.ones(syssize)
Ee_02=-40.7388
spatial_clamp = 6.0

In [14]:
seed=111
set_cpu_deterministic(seed)
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzGRU', units=units,spatial_clamp=spatial_clamp, seed=seed)
h_wl = f'J2={J2_}_sc={spatial_clamp}/seed={seed}/N100_J1=1.0|J2={J2_}_100_LorentzGRU_70_ns=80_MsTrue_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf, numsamples, h_wl, Ee_02, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 15,615
Successfully remapped and loaded weights.
Clipped 528 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-40.04059982299805-0.0010999999940395355j), var E = 0.30300000309944153
DMRG energy  is -40.7388
Time taken=0.778 hrs


In [5]:
seed=222
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzGRU', units=units,spatial_clamp=spatial_clamp, seed=seed)
h_wl = f'{fname}/J2={J2_}_sc={spatial_clamp}/seed={seed}/N100_J1=1.0|J2={J2_}_100_LorentzGRU_70_ns=80_MsTrue_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf, numsamples, h_wl, Ee_02, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 15,615
Successfully remapped and loaded weights.
Clipped 398 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-39.60449981689453-0.0031999999191612005j), var E = 0.896399974822998
DMRG energy  is -40.7388
Time taken=0.883 hrs


In [6]:
seed=333
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzGRU', units=units,spatial_clamp=spatial_clamp, seed=seed)
h_wl = f'{fname}/J2={J2_}_sc={spatial_clamp}/seed={seed}/N100_J1=1.0|J2={J2_}_100_LorentzGRU_70_ns=80_MsTrue_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf, numsamples, h_wl, Ee_02, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 15,615
Successfully remapped and loaded weights.
Clipped 239 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-39.39179992675781+0.003700000001117587j), var E = 0.21729999780654907
DMRG energy  is -40.7388
Time taken=1.023 hrs


In [7]:
seed=444
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzGRU', units=units,spatial_clamp=spatial_clamp, seed=seed)
h_wl = f'{fname}/J2={J2_}_sc={spatial_clamp}/seed={seed}/N100_J1=1.0|J2={J2_}_100_LorentzGRU_70_ns=80_MsTrue_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf, numsamples, h_wl, Ee_02, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 15,615
Successfully remapped and loaded weights.
Clipped 125 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-40.276100158691406+0.0013000000035390258j), var E = 0.13950000703334808
DMRG energy  is -40.7388
Time taken=1.033 hrs


In [9]:
seed=555
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzGRU', units=units,spatial_clamp=spatial_clamp, seed=seed)
h_wl = f'J2={J2_}_sc={spatial_clamp}/seed={seed}/N100_J1=1.0|J2={J2_}_100_LorentzGRU_70_ns=80_MsTrue_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf, numsamples, h_wl, Ee_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 15,615
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-40.31529998779297+0.003700000001117587j), var E = 0.7436000108718872
DMRG energy  is -40.7388
Time taken=1.286 hrs


## J2=0.5, spatial clamp=4.0

In [15]:
J2_ = 0.5
J2 = +J2_ * np.ones(syssize)
Ee_05=37.5000
spatial_clamp=4.0

In [17]:
seed=111
set_cpu_deterministic(seed)
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzGRU', units=units,spatial_clamp=spatial_clamp, seed=seed)
h_wl = f'J2={J2_}_sc={spatial_clamp}/seed={seed}/N100_J1=1.0|J2={J2_}_100_LorentzGRU_70_ns=80_MsTrue_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf, numsamples, h_wl, Ee_05, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 15,615
Successfully remapped and loaded weights.
Clipped 101 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-37.48899841308594-0.000699999975040555j), var E = 0.02500000037252903
DMRG energy  is 37.5
Time taken=0.954 hrs


In [12]:
seed=222
set_cpu_deterministic(seed)
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzGRU', units=units,spatial_clamp=spatial_clamp, seed=seed)
h_wl = f'{fname}/J2={J2_}_sc={spatial_clamp}/seed={seed}/N100_J1=1.0|J2={J2_}_100_LorentzGRU_70_ns=80_MsTrue_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf, numsamples, h_wl, Ee_05, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 15,615
Successfully remapped and loaded weights.
Clipped 368 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-36.97439956665039-0.0005000000237487257j), var E = 0.06509999930858612
DMRG energy  is 37.5
Time taken=1.11 hrs


In [14]:
seed=333
set_cpu_deterministic(seed)
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzGRU', units=units,spatial_clamp=spatial_clamp, seed=seed)
h_wl = f'{fname}/J2={J2_}_sc={spatial_clamp}/seed={seed}/N100_J1=1.0|J2={J2_}_100_LorentzGRU_70_ns=80_MsTrue_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf, numsamples, h_wl, Ee_05, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 15,615
Successfully remapped and loaded weights.
Clipped 125 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-36.895599365234375+0.00039999998989515007j), var E = 0.2969000041484833
DMRG energy  is 37.5
Time taken=1.102 hrs
